# カテゴリ1 チーム基礎実力（Team Baseline）

##  Jリーグ公式から2021年~2025年の試合結果を取得

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from typing import Dict, List


def scrape_jleague_fixtures(
    year_id_map: Dict[int, int],
    delay: float = 1.0,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Jリーグ公式サイトから試合結果データを取得する

    Parameters
    ----------
    year_id_map : Dict[int, int]
        年と大会IDのマッピング（例: {2021: 492, 2022: 521, ...}）
    delay : float, default 1.0
        リクエスト間の待機時間（秒）
    verbose : bool, default True
        進捗メッセージを表示するかどうか

    Returns
    -------
    pd.DataFrame
        試合結果データ（列: Season, Competition, Section, Date, Home, Score,
        Away, Stadium, Attendance, Home_Goals, Away_Goals, Date_Parsed）
    """
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/91.0.4472.124 Safari/537.36"
        )
    }

    all_data: List[dict] = []
    
    if verbose:
        print("データ取得を開始します...")

    for year, comp_id in year_id_map.items():
        if verbose:
            print(f"--- {year}年 (ID: {comp_id}) を取得中 ---")

        url = (
            "https://data.j-league.or.jp/SFMS01/search?"
            f"competition_years={year}&competition_frame_ids=1"
            f"&competition_ids={comp_id}&tv_relay_station_name="
        )

        try:
            # ページ取得
            response = requests.get(url, headers=headers)
            response.raise_for_status()
            response.encoding = response.apparent_encoding  # 文字化け防止
            soup = BeautifulSoup(response.text, "html.parser")

            # 試合テーブルの取得
            table = soup.find("table", class_="table-base00")
            if table is None:
                if verbose:
                    print(f"  Warning: {year}年のテーブルが見つかりませんでした。スキップします。")
                continue

            tbody = table.find("tbody") or table
            rows = tbody.find_all("tr")

            year_match_count = 0

            for row in rows:
                cols = row.find_all("td")
                if len(cols) < 10:
                    continue

                # スコア（"2-1" など）を取得
                score_tag = cols[6].find("a")
                score_text = score_tag.text.strip() if score_tag else cols[6].text.strip()

                # 試合中・未開催など "数字-数字" 形式でないものは除外
                if "-" not in score_text:
                    continue

                item = {
                    "Season": cols[0].text.strip(),
                    "Competition": cols[1].text.strip(),
                    "Section": cols[2].text.strip(),
                    "Date": cols[3].text.strip(),
                    "Home": cols[5].text.strip(),
                    "Score": score_text,
                    "Away": cols[7].text.strip(),
                    "Stadium": cols[8].text.strip(),
                    "Attendance": cols[9].text.strip().replace(",", ""),
                }

                # Season 列に対象年が含まれているものだけ採用
                season_str = item["Season"]
                year_str = str(year)
                year_short = year_str[-2:]
                if year_str not in season_str and year_short not in season_str:
                    continue

                all_data.append(item)
                year_match_count += 1

            if verbose:
                print(f"  -> {year_match_count} 件取得完了")
            time.sleep(delay)  # サーバー負荷軽減

        except Exception as e:
            # 年ごとの取得でエラーが出ても、他の年の処理は続ける
            if verbose:
                print(f" Error: {year}年のデータ取得中にエラーが発生しました - {e}")

    # データフレームに変換
    if not all_data:
        if verbose:
            print("\nWarning: いずれの年でも試合データが取得できませんでした。")
        df_all = pd.DataFrame(
            columns=[
                "Season",
                "Competition",
                "Section",
                "Date",
                "Home",
                "Score",
                "Away",
                "Stadium",
                "Attendance",
                "Home_Goals",
                "Away_Goals",
                "Date_Parsed",
            ]
        )
    else:
        df_all = pd.DataFrame(all_data)

        # スコアを分割して数値列を追加
        df_all[["Home_Goals", "Away_Goals"]] = (
            df_all["Score"].str.split("-", expand=True).astype(int)
        )

        # 日付をパースして時系列順にソート
        df_all["Date_Parsed"] = pd.to_datetime(
            df_all["Date"].str.extract(r"(\d{2}/\d{2}/\d{2})")[0], format="%y/%m/%d"
        )
        df_all = df_all.sort_values(["Season", "Date_Parsed"]).reset_index(drop=True)

    if verbose:
        print(f"\n全期間の取得が完了しました。総試合数: {len(df_all)}")

    return df_all


# 実行例
year_id_map = {
    2021: 492,
    2022: 521,
    2023: 554,
    2024: 589,
    2025: 651,
}

df_fixtures = scrape_jleague_fixtures(year_id_map)

display(df_fixtures.head())
display(df_fixtures.tail())

データ取得を開始します...
--- 2021年 (ID: 492) を取得中 ---
  -> 380 件取得完了
--- 2022年 (ID: 521) を取得中 ---
  -> 306 件取得完了
--- 2023年 (ID: 554) を取得中 ---
  -> 306 件取得完了
--- 2024年 (ID: 589) を取得中 ---
  -> 380 件取得完了
--- 2025年 (ID: 651) を取得中 ---
  -> 380 件取得完了

全期間の取得が完了しました。総試合数: 1752


,Season,Competition,Section,Date,Home,Score,Away,Stadium,Attendance,Home_Goals,Away_Goals,Date_Parsed
0,2021,Ｊ１,第１節第１日,21/02/26(金),川崎Ｆ,2-0,横浜FM,等々力,4868,2,0,2021-02-26
1,2021,Ｊ１,第１節第２日,21/02/27(土),浦和,1-1,FC東京,埼玉,4943,1,1,2021-02-27
2,2021,Ｊ１,第１節第２日,21/02/27(土),札幌,5-1,横浜FC,札幌ド,11897,5,1,2021-02-27
3,2021,Ｊ１,第１節第２日,21/02/27(土),大分,1-1,徳島,昭和電ド,7012,1,1,2021-02-27
4,2021,Ｊ１,第１節第２日,21/02/27(土),広島,1-1,仙台,Ｅスタ,8820,1,1,2021-02-27


,Season,Competition,Section,Date,Home,Score,Away,Stadium,Attendance,Home_Goals,Away_Goals,Date_Parsed
1747,2025,Ｊ１,第３８節第１日,25/12/06(土),鹿島,2-1,横浜FM,メルスタ,37079,2,1,2025-12-06
1748,2025,Ｊ１,第３８節第１日,25/12/06(土),清水,1-2,岡山,アイスタ,18346,1,2,2025-12-06
1749,2025,Ｊ１,第３８節第１日,25/12/06(土),名古屋,1-0,福岡,豊田ス,34975,1,0,2025-12-06
1750,2025,Ｊ１,第３８節第１日,25/12/06(土),浦和,4-0,川崎Ｆ,埼玉,47878,4,0,2025-12-06
1751,2025,Ｊ１,第３８節第１日,25/12/06(土),柏,1-0,町田,三協Ｆ柏,14092,1,0,2025-12-06


## Jリーグをまとめたwikipediaから、節ごとの順位と、監督情報を取得

In [2]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import io
import re

def clean_text(text):
    """Wikipediaの脚注 [1] や [注釈 1] などを削除する"""
    if not isinstance(text, str):
        return text
    return re.sub(r"\[.*?\]", "", text).strip()


def scrape_j1_wikipedia_multi_years(
    years: list[int],
    delay: float = 1.0,
    verbose: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    複数年のJ1リーグWikipediaデータを取得する

    Parameters
    ----------
    years : list[int]
        取得する年のリスト（例: [2021, 2022, 2023, 2024, 2025]）
    delay : float, default 1.0
        リクエスト間の待機時間（秒）
    verbose : bool, default True
        進捗メッセージを表示するかどうか

    Returns
    -------
    tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]
        (監督情報（シーズン開始時）, 監督交代情報, 節ごとの順位) のタプル
        各DataFrameには 'Year' 列が追加される
    """
    import time

    def _scrape_single_year(year: int):
        """単年のWikipediaデータを取得する内部関数"""
        url = f"https://ja.wikipedia.org/wiki/{year}年のJ1リーグ"
        headers = {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/91.0.4472.124 Safari/537.36"
            )
        }
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, "html.parser")

        # --- 1. 監督情報（シーズン開始時）の取得 ---
        manager_data = pd.DataFrame()
        # --- 2. 監督交代表の取得 ---
        manager_changes_data = pd.DataFrame()

        tables = soup.find_all("table", class_="wikitable")

        for table in tables:
            # ヘッダーを取得
            headers_text = [th.get_text(strip=True) for th in table.find_all("th")]

            # 「チーム名」と「監督」の両方が含まれ、「前監督」が含まれないテーブル（参加クラブの表）
            if "チーム名" in headers_text and "監督" in headers_text and "前監督" not in headers_text:
                try:
                    df_club = pd.read_html(io.StringIO(str(table)), header=0)[0]

                    # 列名の正規化
                    team_col = None
                    manager_col = None
                    for c in df_club.columns:
                        c_str = str(c)
                        if any(x in c_str for x in ["チーム名", "チーム", "クラブ"]):
                            team_col = c
                        if "監督" in c_str and "前監督" not in c_str:
                            manager_col = c

                    if team_col and manager_col:
                        manager_data = df_club[[team_col, manager_col]].copy()
                        manager_data.columns = ["Team", "Manager"]

                        # データのクレンジング
                        manager_data["Team"] = manager_data["Team"].apply(clean_text)
                        manager_data["Manager"] = manager_data["Manager"].apply(clean_text)
                except Exception as e:
                    continue

            # 監督交代表: 「チーム名」「前監督」「退任日」「新監督」「就任日」が含まれるテーブル
            elif "チーム名" in headers_text and "前監督" in headers_text and "新監督" in headers_text:
                try:
                    df_changes = pd.read_html(io.StringIO(str(table)), header=0)[0]

                    # 必要な列を抽出（存在する列のみ）
                    cols_to_keep = []
                    col_mapping = {}

                    for c in df_changes.columns:
                        c_str = str(c)
                        if "チーム名" in c_str or ("チーム" in c_str and "前" not in c_str):
                            col_mapping[c] = "Team"
                            cols_to_keep.append(c)
                        elif "前監督" in c_str:
                            col_mapping[c] = "Previous_Manager"
                            cols_to_keep.append(c)
                        elif "退任日" in c_str:
                            col_mapping[c] = "Resignation_Date"
                            cols_to_keep.append(c)
                        elif "監督代行" in c_str:
                            col_mapping[c] = "Interim_Manager"
                            cols_to_keep.append(c)
                        elif "新監督" in c_str:
                            col_mapping[c] = "New_Manager"
                            cols_to_keep.append(c)
                        elif "就任日" in c_str:
                            col_mapping[c] = "Appointment_Date"
                            cols_to_keep.append(c)
                        elif "備考" in c_str:
                            col_mapping[c] = "Notes"
                            cols_to_keep.append(c)

                    if cols_to_keep:
                        manager_changes_data = df_changes[cols_to_keep].copy()
                        manager_changes_data.rename(columns=col_mapping, inplace=True)

                        # データのクレンジング
                        for col in manager_changes_data.columns:
                            manager_changes_data[col] = manager_changes_data[col].apply(clean_text)
                except Exception as e:
                    continue

        # --- 3. 節ごとの順位（順位推移表）の取得 ---
        rank_tables = soup.find_all("table", class_=lambda x: x and "sportsrbrtable" in x)

        all_rank_dfs = []
        for table in rank_tables:
            try:
                df_step = pd.read_html(io.StringIO(str(table)), header=0)[0]

                # マルチインデックスの解消
                if isinstance(df_step.columns, pd.MultiIndex):
                    df_step.columns = df_step.columns.get_level_values(-1)

                # 最初の列（「チーム ╲ 節」など）を「Team」に書き換え
                if len(df_step.columns) > 0:
                    df_step.columns = ["Team"] + list(df_step.columns[1:])

                    # チーム名のクレンジング
                    df_step["Team"] = df_step["Team"].apply(clean_text)
                    all_rank_dfs.append(df_step)
            except Exception as e:
                continue

        # 複数の表（前半戦・後半戦など）がある場合に結合
        if len(all_rank_dfs) >= 2:
            df_ranks = all_rank_dfs[0]
            for next_df in all_rank_dfs[1:]:
                # 重複する列（Team以外）を排除して結合
                cols_to_use = next_df.columns.difference(df_ranks.columns).tolist()
                if cols_to_use:
                    df_ranks = pd.merge(
                        df_ranks, next_df[["Team"] + cols_to_use], on="Team", how="outer"
                    )
        elif len(all_rank_dfs) == 1:
            df_ranks = all_rank_dfs[0]
        else:
            df_ranks = pd.DataFrame()

        return manager_data, manager_changes_data, df_ranks

    all_managers = []
    all_manager_changes = []
    all_ranks = []

    if verbose:
        print(f"Wikipediaから {len(years)} 年分のデータを取得します...")

    for year in years:
        if verbose:
            print(f"\n--- {year}年 を取得中 ---")

        try:
            managers, manager_changes, ranks = _scrape_single_year(year)

            # 年を追加
            if not managers.empty:
                managers["Year"] = year
                all_managers.append(managers)
                if verbose:
                    print(f"  監督情報: {len(managers)} 件")

            if not manager_changes.empty:
                manager_changes["Year"] = year
                all_manager_changes.append(manager_changes)
                if verbose:
                    print(f"  監督交代情報: {len(manager_changes)} 件")

            if not ranks.empty:
                ranks["Year"] = year
                all_ranks.append(ranks)
                if verbose:
                    print(f"  順位推移表: {len(ranks)} チーム × {len(ranks.columns) - 2} 節")

            time.sleep(delay)  # サーバー負荷軽減

        except Exception as e:
            if verbose:
                print(f"  Error: {year}年のデータ取得中にエラーが発生しました - {e}")
            continue

    # データを結合
    df_managers_all = (
        pd.concat(all_managers, ignore_index=True) if all_managers else pd.DataFrame()
    )
    df_manager_changes_all = (
        pd.concat(all_manager_changes, ignore_index=True)
        if all_manager_changes
        else pd.DataFrame()
    )
    df_ranks_all = (
        pd.concat(all_ranks, ignore_index=True) if all_ranks else pd.DataFrame()
    )

    if verbose:
        print(f"\n全期間の取得が完了しました。")
        print(f"  監督情報: {len(df_managers_all)} 件")
        print(f"  監督交代情報: {len(df_manager_changes_all)} 件")
        print(f"  順位推移表: {len(df_ranks_all)} 行")

    return df_managers_all, df_manager_changes_all, df_ranks_all


# 実行例: 2021〜2025年のデータを取得
years = [2021, 2022, 2023, 2024, 2025]
df_managers, df_manager_changes, df_ranks = scrape_j1_wikipedia_multi_years(years)

print("\n=== 監督情報（シーズン開始時）===")
if not df_managers.empty:
    display(df_managers.head(10))
    print(f"\n総件数: {len(df_managers)} 件")
else:
    print("監督情報が見つかりませんでした。")

print("\n=== 監督交代情報 ===")
if not df_manager_changes.empty:
    display(df_manager_changes)
    print(f"\n総件数: {len(df_manager_changes)} 件")
else:
    print("監督交代情報が見つかりませんでした。")

print("\n=== 節ごとの順位（最初の5チーム・抜粋）===")
if not df_ranks.empty:
    # 表示用に最初の数節のみ出力
    display(df_ranks.iloc[:5, :12])
    print(f"\n総行数: {len(df_ranks)} 行")
else:
    print("順位表が見つかりませんでした。")

Wikipediaから 5 年分のデータを取得します...

--- 2021年 を取得中 ---
  監督情報: 20 件
  監督交代情報: 10 件
  順位推移表: 21 チーム × 38 節

--- 2022年 を取得中 ---
  監督情報: 18 件
  監督交代情報: 6 件
  順位推移表: 19 チーム × 34 節

--- 2023年 を取得中 ---
  監督情報: 18 件
  監督交代情報: 2 件
  順位推移表: 19 チーム × 34 節

--- 2024年 を取得中 ---
  監督情報: 20 件
  監督交代情報: 4 件
  順位推移表: 21 チーム × 38 節

--- 2025年 を取得中 ---
  監督情報: 20 件
  監督交代情報: 4 件
  順位推移表: 21 チーム × 38 節

全期間の取得が完了しました。
  監督情報: 96 件
  監督交代情報: 26 件
  順位推移表: 101 行

=== 監督情報（シーズン開始時）===


,Team,Manager,Year
0,北海道コンサドーレ札幌,ペトロヴィッチ,2021
1,ベガルタ仙台,手倉森誠,2021
2,鹿島アントラーズ,ザーゴ,2021
3,浦和レッズ,リカルド・ロドリゲス,2021
4,柏レイソル,ネルシーニョ,2021
5,FC東京,長谷川健太,2021
6,川崎フロンターレ,鬼木達,2021
7,横浜F・マリノス,アンジェ・ポステコグルー,2021
8,横浜FC,下平隆宏,2021
9,湘南ベルマーレ,浮嶋敏,2021



総件数: 96 件

=== 監督交代情報 ===


,Team,Previous_Manager,Resignation_Date,Interim_Manager,New_Manager,Appointment_Date,Notes,Year
0,横浜FC,下平隆宏,4月7日,NaN,早川知伸,4月8日,ユース監督からの異動,2021
1,鹿島アントラーズ,ザーゴ,4月14日,NaN,相馬直樹,4月14日,コーチからの昇格,2021
2,ガンバ大阪,宮本恒靖,5月13日,松波正信（強化アカデミー部長）,松波正信,6月1日,強化アカデミー部長からの異動,2021
3,横浜F・マリノス,アンジェ・ポステコグルー,6月10日,松永英機（アカデミーダイレクター）,ケヴィン・マスカット,8月5日,外部からの招聘,2021
4,セレッソ大阪,レヴィー・クルピ,8月26日,NaN,小菊昭雄,8月26日,コーチからの昇格,2021
5,湘南ベルマーレ,浮嶋敏,8月31日,NaN,山口智,9月1日,コーチからの昇格,2021
6,サンフレッチェ広島,城福浩,10月25日,NaN,沢田謙太郎,10月26日,ヘッドコーチからの昇格,2021
7,清水エスパルス,ロティーナ,11月4日,NaN,平岡宏章,11月4日,コーチからの昇格,2021
8,FC東京,長谷川健太,11月7日,NaN,森下申一,11月10日,GKコーチからの異動,2021
9,ベガルタ仙台,手倉森誠,11月23日,NaN,原崎政人,11月23日,ヘッドコーチからの昇格,2021



総件数: 26 件

=== 節ごとの順位（最初の5チーム・抜粋）===


,Team,1,2,3,4,5,6,7,8,9,10,11
0,FC東京,8.0,5.0,9.0,10.0,7.0,6.0,7.0,6.0,8.0,8.0,10.0
1,アビスパ福岡,14.0,13.0,16.0,12.0,9.0,10.0,10.0,11.0,11.0,11.0,9.0
2,ガンバ大阪,15.0,17.0,18.0,19.0,19.0,19.0,18.0,18.0,18.0,17.0,17.0
3,サガン鳥栖,7.0,2.0,2.0,3.0,3.0,3.0,4.0,7.0,4.0,3.0,3.0
4,サンフレッチェ広島,8.0,11.0,6.0,7.0,6.0,5.0,6.0,4.0,6.0,7.0,7.0



総行数: 101 行


## TransferMarketから、各チームの市場価値を取得

In [3]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import random

# 最新のブラウザのUser-Agentを複数用意
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:123.0) Gecko/20100101 Firefox/123.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.3 Safari/605.1.15"
]

def scrape_transfermarkt_market_value(year, max_retries=3):
    """
    指定されたJリーグのシーズン年（year）に対応するTransferMarktデータを取得する
    失敗した場合は最大 max_retries 回まで自動的に再試行する
    """
    saison_id = year - 1
    url = f"https://www.transfermarkt.com/j1-league/startseite/wettbewerb/JAP1/plus/?saison_id={saison_id}"
    
    for attempt in range(max_retries):
        # アクセスごとにブラウザ情報と、人間らしい複雑なヘッダーを構築
        headers = {
            "User-Agent": random.choice(USER_AGENTS),
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.5",
            "Connection": "keep-alive",
            "Upgrade-Insecure-Requests": "1"
        }

        try:
            response = requests.get(url, headers=headers, timeout=15)
            
            # ボット検知(403)やアクセス過多(429)で弾かれた場合の処理
            if response.status_code in [403, 429]:
                wait_time = (2 ** attempt) + random.uniform(2, 4) # 失敗するたびに待機時間を長くする
                print(f"    [Warning] Status {response.status_code}. ブロックされました。{wait_time:.1f}秒待機して再試行します ({attempt+1}/{max_retries})")
                time.sleep(wait_time)
                continue # 次のループ（再試行）へ
                
            response.raise_for_status()
            
            # 正常に取得できた場合はHTML解析へ進むためループを抜ける
            break
            
        except requests.exceptions.RequestException as e:
            print(f"    [Error] 通信エラー: {e}")
            if attempt == max_retries - 1:
                print(f"  [Failed] {year}年の取得をスキップします。")
                return pd.DataFrame() # 全ての試行が失敗したら空のDFを返す
            
            time.sleep(random.uniform(3, 5))
            continue

    # --- HTMLの解析（元のコードと同じロジック） ---
    soup = BeautifulSoup(response.content, 'html.parser')
    table = soup.find('table', class_='items')
    
    if not table:
        print(f"    [Error] {year}年のページにテーブルが見つかりません。")
        return pd.DataFrame()
    
    data = []
    rows = table.find('tbody').find_all('tr', recursive=False)
    
    for row in rows:
        team_td = row.find('td', class_='hauptlink')
        if not team_td:
            continue
        team_name = team_td.get_text(strip=True)
        
        value_tds = row.find_all('td', class_='rechts')
        if not value_tds:
            continue
        
        value_text = value_tds[-1].get_text(strip=True)
        
        numeric_value = 0.0
        val_clean = value_text.replace('€', '').replace(',', '')
        
        try:
            if 'm' in val_clean:
                numeric_value = float(val_clean.replace('m', ''))
            elif 'k' in val_clean:
                numeric_value = float(val_clean.replace('k', '')) / 1000.0
        except ValueError:
            numeric_value = 0.0
            
        data.append({
            'Season': year,
            'Team_TM': team_name, 
            'TotalMarketValue_M_Euro': numeric_value
        })
    
    return pd.DataFrame(data)

# --- 複数年の取得実行 ---
years = [2021, 2022, 2023, 2024, 2025]
all_tm_data = []

print("TransferMarktから市場価値データを取得中...")

for y in years:
    print(f"--- {y}年シーズンを取得中 ---")
    df_year = scrape_transfermarkt_market_value(y)
    
    if not df_year.empty:
        all_tm_data.append(df_year)
        print(f"  -> {len(df_year)} チーム取得完了")
    
    # リクエスト成功後も、相手サーバーを警戒させないために3〜6秒のランダムな待機を入れる
    sleep_time = random.uniform(3.0, 6.0)
    time.sleep(sleep_time)

# ひとつのDataFrameに結合
if all_tm_data:
    df_market_values = pd.concat(all_tm_data, ignore_index=True)
    print("\n全期間の取得が完了しました。")
    display(df_market_values.head())
else:
    print("データが取得できませんでした。")

TransferMarktから市場価値データを取得中...
--- 2021年シーズンを取得中 ---
  -> 20 チーム取得完了
--- 2022年シーズンを取得中 ---
  -> 18 チーム取得完了
--- 2023年シーズンを取得中 ---
  -> 18 チーム取得完了
--- 2024年シーズンを取得中 ---
  -> 20 チーム取得完了
--- 2025年シーズンを取得中 ---
  -> 20 チーム取得完了

全期間の取得が完了しました。


,Season,Team_TM,TotalMarketValue_M_Euro
0,2021,Urawa Red Diamonds,25.35
1,2021,Vissel Kobe,25.03
2,2021,Kawasaki Frontale,24.15
3,2021,Yokohama F. Marinos,21.53
4,2021,Kashima Antlers,21.43


# 2. チームの勢い＆直接対決（Team Momentum & H2H）

## Football Labからゴール期待値と守備評価（KAGI）、攻撃指標（AGI）を取得

In [4]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time

def scrape_football_lab_xg(year):
    """
    指定されたシーズンのFootball-Labからチームごとのゴール期待値をスクレイピングする
    """
    url = f"https://www.football-lab.jp/summary/team_ranking/j1/?year={year}&data=expected"
    
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/91.0.4472.124 Safari/537.36"
        )
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"  [Error] {year}年の取得に失敗しました: {e}")
        return pd.DataFrame()

    soup = BeautifulSoup(response.content, 'html.parser')
    
    # データが格納されているテーブルを取得
    table = soup.find('table', class_='statsTbl')
    if not table:
        print(f"  [Warning] {year}年のデータテーブルが見つかりませんでした。")
        return pd.DataFrame()
    
    data = []
    # tbody内の各行（チームごとのデータ）を処理
    rows = table.find('tbody').find_all('tr')
    
    for row in rows:
        cols = row.find_all('td')
        if len(cols) < 6:
            continue
            
        # 1. チーム名の取得（.dsktp クラスからフルネームを抽出）
        name_td = cols[1]
        dsktp_span = name_td.find('span', class_='dsktp')
        if dsktp_span:
            team_name = dsktp_span.get_text(strip=True)
        else:
            team_name = name_td.get_text(strip=True)
            
        # 2. 各種数値データの取得（文字列からfloat/intに変換）
        try:
            xg_value = float(cols[2].get_text(strip=True))      # 期待値
            actual_goals = float(cols[3].get_text(strip=True))  # ゴール
            diff = float(cols[4].get_text(strip=True))          # 差分
            rank = int(cols[5].get_text(strip=True))            # 成績順位
        except ValueError:
            continue
            
        data.append({
            'Season': year,
            'Team_FL': team_name,
            'Expected_Goals': xg_value,
            'Actual_Goals': actual_goals,
            'xG_Diff': diff,
            'Rank_FL': rank
        })
        
    return pd.DataFrame(data)

# --- 複数年の取得実行 ---
years = [2021, 2022, 2023, 2024, 2025]
all_fl_data = []

print("Football-Labからゴール期待値データを取得中...")

for y in years:
    print(f"--- {y}年シーズンを取得中 ---")
    df_year = scrape_football_lab_xg(y)
    if not df_year.empty:
        all_fl_data.append(df_year)
        print(f"  -> {len(df_year)} チーム取得完了")
    
    # サーバー負荷軽減のためのインターバル
    time.sleep(1.5)

# ひとつのDataFrameに結合
if all_fl_data:
    df_xg = pd.concat(all_fl_data, ignore_index=True)
    print("\n全期間の取得が完了しました。")
    display(df_xg.head(10))
else:
    print("データが取得できませんでした。")

Football-Labからゴール期待値データを取得中...
--- 2021年シーズンを取得中 ---
  -> 20 チーム取得完了
--- 2022年シーズンを取得中 ---
  -> 18 チーム取得完了
--- 2023年シーズンを取得中 ---
  -> 18 チーム取得完了
--- 2024年シーズンを取得中 ---
  -> 20 チーム取得完了
--- 2025年シーズンを取得中 ---
  -> 20 チーム取得完了

全期間の取得が完了しました。


,Season,Team_FL,Expected_Goals,Actual_Goals,xG_Diff,Rank_FL
0,2021,横浜Ｆ・マリノス,1.876,2.1,0.224,2
1,2021,川崎フロンターレ,1.710,2.1,0.390,1
2,2021,鹿島アントラーズ,1.585,1.6,0.015,4
3,2021,北海道コンサドーレ札幌,1.530,1.2,-0.330,10
4,2021,セレッソ大阪,1.433,1.2,-0.233,12
5,2021,ヴィッセル神戸,1.333,1.6,0.267,3
6,2021,柏レイソル,1.294,1.0,-0.294,15
7,2021,浦和レッズ,1.284,1.2,-0.084,6
8,2021,サンフレッチェ広島,1.282,1.2,-0.082,11
9,2021,ＦＣ東京,1.241,1.3,0.059,9


In [5]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time

def scrape_football_lab_agi_kagi(year):
    """
    指定されたシーズンのFootball-LabからAGI(攻撃)とKAGI(守備)の指標を取得する
    """
    # AGI/KAGIが掲載されているURL
    url = f"https://www.football-lab.jp/summary/team_ranking/j1?year={year}&data=kagi"
    
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/91.0.4472.124 Safari/537.36"
        )
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"  [Error] {year}年の取得に失敗しました: {e}")
        return pd.DataFrame()

    soup = BeautifulSoup(response.content, 'html.parser')
    
    # statsTblクラスを持つテーブルをすべて取得 (通常 0番目がAGI, 1番目がKAGI)
    tables = soup.find_all('table', class_='statsTbl')
    if len(tables) < 2:
        print(f"  [Warning] {year}年のAGI/KAGIテーブルが正しく見つかりませんでした。")
        return pd.DataFrame()

    def parse_table(table, metric_name):
        res_data = {}
        rows = table.find('tbody').find_all('tr')
        for row in rows:
            cols = row.find_all('td')
            if len(cols) < 5: continue
            
            # チーム名 (dsktpクラス)
            dsktp_span = cols[2].find('span', class_='dsktp')
            team_name = dsktp_span.get_text(strip=True) if dsktp_span else cols[2].get_text(strip=True)
            
            # 指標の値 (3番目のtd)
            try:
                val = float(cols[3].get_text(strip=True))
            except ValueError:
                val = 0.0
            
            res_data[team_name] = val
        return res_data

    # 各テーブルからデータを辞書形式で抽出
    agi_map = parse_table(tables[0], "AGI")
    kagi_map = parse_table(tables[1], "KAGI")

    # データを結合してリスト化
    combined_data = []
    # AGIテーブルにあるチーム名を基準に回す
    for team, agi_val in agi_map.items():
        combined_data.append({
            'Season': year,
            'Team_FL': team,
            'AGI': agi_val,
            'KAGI': kagi_map.get(team, 0.0) # KAGIテーブルから対応する値を取得
        })
        
    return pd.DataFrame(combined_data)

# --- 複数年の取得実行 ---
years = [2021, 2022, 2023, 2024, 2025]
all_agi_kagi_data = []

print("Football-LabからAGI/KAGIデータを取得中...")

for y in years:
    print(f"--- {y}年シーズンを取得中 ---")
    df_year = scrape_football_lab_agi_kagi(y)
    if not df_year.empty:
        all_agi_kagi_data.append(df_year)
        print(f"  -> {len(df_year)} チーム取得完了")
    
    time.sleep(1.5)

# ひとつのDataFrameに結合
if all_agi_kagi_data:
    df_agi_kagi = pd.concat(all_agi_kagi_data, ignore_index=True)
    print("\n全期間の取得が完了しました。")
    display(df_agi_kagi.head(10))
else:
    print("データが取得できませんでした。")

Football-LabからAGI/KAGIデータを取得中...
--- 2021年シーズンを取得中 ---
  -> 20 チーム取得完了
--- 2022年シーズンを取得中 ---
  -> 18 チーム取得完了
--- 2023年シーズンを取得中 ---
  -> 18 チーム取得完了
--- 2024年シーズンを取得中 ---
  -> 20 チーム取得完了
--- 2025年シーズンを取得中 ---
  -> 20 チーム取得完了

全期間の取得が完了しました。


,Season,Team_FL,AGI,KAGI
0,2021,川崎フロンターレ,59.4,54.1
1,2021,鹿島アントラーズ,57.5,58.5
2,2021,横浜Ｆ・マリノス,55.8,56.0
3,2021,サンフレッチェ広島,54.4,54.6
4,2021,ＦＣ東京,52.6,47.9
5,2021,湘南ベルマーレ,52.6,51.1
6,2021,北海道コンサドーレ札幌,52.4,53.7
7,2021,柏レイソル,51.3,53.5
8,2021,アビスパ福岡,50.4,48.2
9,2021,セレッソ大阪,50.1,47.6


## 日本のサッカー競技場一覧 Wikipediaからスタジアムの収容キャパを取得

In [6]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re

def scrape_wikipedia_stadium_capacity():
    """
    Wikipediaの「日本のサッカー競技場一覧」からスタジアム名と入場可能数を取得する
    """
    url = f"https://ja.wikipedia.org/wiki/日本のサッカー競技場一覧"
    
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/91.0.4472.124 Safari/537.36"
        )
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"  [Error] データの取得に失敗しました: {e}")
        return pd.DataFrame()

    soup = BeautifulSoup(response.content, 'html.parser')
    
    # 「wikitable sortable」クラスを持つテーブルを取得
    # 一覧ページには複数のテーブルがある可能性があるため、ヘッダー内容で対象を特定
    target_table = None
    tables = soup.find_all('table', class_='wikitable')
    for table in tables:
        header_text = table.get_text()
        if 'スタジアム名称' in header_text and '入場可能数' in header_text:
            target_table = table
            break

    if not target_table:
        print("  [Warning] 対象のスタジアムテーブルが見つかりませんでした。")
        return pd.DataFrame()

    stadium_data = []
    rows = target_table.find('tbody').find_all('tr')

    for row in rows:
        cols = row.find_all(['td', 'th'])
        # ヘッダー行をスキップ、または列数が足りない行をスキップ
        if len(cols) < 5 or cols[0].name == 'th':
            continue
            
        # --- スタジアム名称の抽出 ---
        # 命名権名称などが含まれるため、テキスト全体を取得して改行などを整理
        name_cell = cols[1]
        stadium_name = name_cell.get_text(" ", strip=True)

        # --- 入場可能数の抽出 ---
        cap_cell = cols[2]
        
        # Wikipedia特有のソート用非表示スパン（<span style="display:none">...</span>）を削除
        for span in cap_cell.find_all("span", style=lambda x: x and ("hidden" in x or "none" in x)):
            span.decompose()
            
        cap_text = cap_cell.get_text(strip=True)
        
        # カンマを除去して数値に変換
        try:
            # 数字以外の文字（カンマや注釈など）を除去して整数化
            clean_cap = re.sub(r'[^\d]', '', cap_text)
            capacity = int(clean_cap) if clean_cap else 0
        except ValueError:
            capacity = 0
            
        # --- 住所/所在地 ---
        address = cols[3].get_text(" ", strip=True)
        
        # クラブの取得
        club = cols[4].get_text(strip=True)

        stadium_data.append({
            'Stadium_Name': stadium_name,
            'Capacity': capacity,
            'Address': address,
            'Club': club
        })
        
    return pd.DataFrame(stadium_data)

# --- 実行 ---
print("Wikipediaからスタジアムデータを取得中...")
stadium_df = scrape_wikipedia_stadium_capacity()

if not stadium_df.empty:
    print(f"取得完了: {len(stadium_df)} 件")
    # 入場可能数でソートして上位を表示
    display(stadium_df.sort_values('Capacity', ascending=False).head(10))
else:
    print("データが取得できませんでした。")

from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="jleague_stadium_search")

def get_lat_lon(address):
    try:
        # 住所から座標を検索
        location = geolocator.geocode(address)
        if location:
            return location.latitude, location.longitude
        else:
            return None, None
    except Exception as e:
        print(f"Error: {e}")
        return None, None
    finally:
        # サーバー負荷軽減のため1秒待機（Nominatimの利用規約）
        time.sleep(1)

# 新しい列として追加
print("緯度経度を取得中...")
stadium_df[['Latitude', 'Longitude']] = stadium_df['Address'].apply(
    lambda x: pd.Series(get_lat_lon(x))
)

display(stadium_df)

Wikipediaからスタジアムデータを取得中...
取得完了: 58 件


,Stadium_Name,Capacity,Address,Club
19,横浜国際総合競技場 （ 日産 スタジアム）,71624,神奈川県 横浜市 港北区,横浜F・マリノス
12,埼玉スタジアム2002,62040,埼玉県 さいたま市 緑区,浦和レッズ
16,東京スタジアム （ 味の素 スタジアム）,47851,東京都 調布市,FC東京
32,豊田スタジアム,42753,愛知県 豊田市,名古屋グランパス
26,新潟スタジアム （ デンカ ビッグスワンスタジアム）,41684,新潟県 新潟市 中央区,アルビレックス新潟
35,市立吹田サッカースタジアム （ パナソニック スタジアム 吹田）,39694,大阪府 吹田市,ガンバ大阪
7,茨城県立カシマサッカースタジアム （ メルカリ スタジアム）,39095,茨城県 鹿嶋市,鹿島アントラーズ
0,札幌ドーム （ 大和ハウス プレミスト ドーム）,38794,北海道 札幌市 豊平区,北海道コンサドーレ札幌
54,大分スポーツ公園総合競技場 （ クラサス ドーム大分）,31997,大分県 大分市,大分トリニータ
53,熊本県民総合運動公園陸上競技場 （ えがお 健康スタジアム）,30275,熊本県 熊本市 東区,ロアッソ熊本


緯度経度を取得中...


,Stadium_Name,Capacity,Address,Club,Latitude,Longitude
0,札幌ドーム （ 大和ハウス プレミスト ドーム）,38794,北海道 札幌市 豊平区,北海道コンサドーレ札幌,43.031242,141.380050
1,八戸市多賀多目的運動場 （ プライフーズ スタジアム）,5124,青森県 八戸市,ヴァンラーレ八戸,40.512239,141.488296
2,仙台スタジアム （ ユアテック スタジアム仙台）,19526,宮城県 仙台市 泉区,ベガルタ仙台,38.326355,140.881551
3,秋田市八橋運動公園陸上競技場 （ ソユー スタジアム）,18560,秋田県 秋田市,ブラウブリッツ秋田,39.719763,140.103467
4,山形県総合運動公園陸上競技場 （ NDソフト スタジアム山形）,20638,山形県 天童市,モンテディオ山形,38.362434,140.377263
5,福島県営あづま陸上競技場 （ とうほう ・みんなのスタジアム）,5710,福島県 福島市,福島ユナイテッドFC,37.760777,140.474581
6,いわきグリーンフィールド （ ハワイアンズ スタジアムいわき）,5066,福島県 いわき市,いわきFC,37.050423,140.887634
7,茨城県立カシマサッカースタジアム （ メルカリ スタジアム）,39095,茨城県 鹿嶋市,鹿島アントラーズ,35.966116,140.645029
8,水戸市立競技場 （ ケーズデンキ スタジアム水戸）,10152,茨城県 水戸市,水戸ホーリーホック,36.365917,140.473174
9,栃木県総合運動公園陸上競技場 （ カンセキ スタジアムとちぎ）,24670,栃木県 宇都宮市,栃木SC,36.554968,139.882878


# 3. 個人の稼働状況＆期待値（Player Availability & xG）

In [7]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

# 1. 設定
YEARS = [2021, 2022, 2023, 2024, 2025]
TEAMS = {
    "kasm": "鹿島アントラーズ", "uraw": "浦和レッズ", "kasw": "柏レイソル",
    "tk-v": "東京ヴェルディ", "ka-f": "川崎フロンターレ", "shim": "清水エスパルス",
    "kyot": "京都サンガF.C.", "c-os": "セレッソ大阪", "okay": "ファジアーノ岡山",
    "fuku": "アビスパ福岡", "mito": "水戸ホーリーホック", "chib": "ジェフユナイテッド千葉",
    "fctk": "ＦＣ東京", "mcd": "ＦＣ町田ゼルビア", "y-fm": "横浜Ｆ・マリノス",
    "nago": "名古屋グランパス", "g-os": "ガンバ大阪", "kobe": "ヴィッセル神戸",
    "hiro": "サンフレッチェ広島", "ngsk": "Ｖ・ファーレン長崎", "y-fc": "横浜ＦＣ",
    "shon": "湘南ベルマーレ", "niig": "アルビレックス新潟", "iwat": "ジュビロ磐田",
    "sapp": "北海道コンサドーレ札幌", "tosu": "サガン鳥栖", "toku": "徳島ヴォルティス",
    "oita": "大分トリニータ", "send": "ベガルタ仙台"
}

def get_player_id(href):
    """URLから選手IDを抽出 (/player/1400359/ -> 1400359)"""
    match = re.search(r'/player/(\d+)/', href)
    return match.group(1) if match else None

def scrape_team_stats(team_code, year):
    url = f"https://www.football-lab.jp/{team_code}/ranking?year={year}"
    print(f"Scraping: {TEAMS[team_code]} ({year})")
    
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        soup = BeautifulSoup(r.content, 'html.parser')
    except Exception as e:
        print(f"Error skipping {team_code} {year}: {e}")
        return pd.DataFrame()

    def parse_table(table_selector):
        data = []
        target_table = soup.select_one(table_selector)
        if not target_table:
            return pd.DataFrame()
            
        rows = target_table.find('tbody').find_all('tr')
        for row in rows:
            cols = row.find_all('td')
            if len(cols) < 5: continue
            
            p_link = cols[2].find('a')
            if not p_link: continue
            
            p_id = get_player_id(p_link.get('href'))
            p_name = p_link.text.strip()
            
            # 共通項目
            record = {
                'player_id': p_id,
                'player_name': p_name,
                'pos': cols[1].text.strip(),
                'points': cols[3].text.strip(),
                'avg_90': cols[4].text.strip(),
                'games': cols[5].text.strip(),
                'goals': cols[6].text.strip() if len(cols) > 6 else "0"
            }
            # アシストはCBP(攻撃)テーブルのみ存在
            if len(cols) > 7:
                record['assists'] = cols[7].text.strip()
                
            data.append(record)
        return pd.DataFrame(data)

    # 1. CBP (攻撃)
    df_cbp = parse_table('div[data-name="cbp"] div[data-id="1"] table.statsTbl')
    # 2. シュートポイント
    df_shoot = parse_table('div[data-name="cbp2"] div[data-id="1"] table.statsTbl')
    # 3. ゴールポイント
    df_goal = parse_table('div[data-name="cbp2"] div[data-id="2"] table.statsTbl')

    if df_cbp.empty and df_shoot.empty and df_goal.empty: 
        return pd.DataFrame()

    # --- 修正箇所：結合時に共通の基本情報をキーにする ---
    res = pd.DataFrame()
    common_keys = ['player_id', 'player_name', 'pos', 'games', 'goals']

    if not df_cbp.empty:
        res = df_cbp.rename(columns={'points': 'cbp_attack', 'avg_90': 'cbp_attack_90'})

    if not df_shoot.empty:
        # 必要なカラムだけを抽出してリネーム
        df_shoot_sub = df_shoot[common_keys + ['points', 'avg_90']].rename(
            columns={'points': 'shoot_point', 'avg_90': 'shoot_point_90'}
        )
        if res.empty:
            res = df_shoot_sub
        else:
            # outer結合で、片方にしかいない選手の名前やポジションも保持する
            res = pd.merge(res, df_shoot_sub, on=common_keys, how='outer')

    if not df_goal.empty:
        df_goal_sub = df_goal[common_keys + ['points', 'avg_90']].rename(
            columns={'points': 'goal_point', 'avg_90': 'goal_point_90'}
        )
        if res.empty:
            res = df_goal_sub
        else:
            res = pd.merge(res, df_goal_sub, on=common_keys, how='outer')

    res['year'] = year
    res['team_name'] = TEAMS[team_code]
    return res

# 2. メインループ実行
all_data = []
for year in YEARS:
    for code in TEAMS.keys():
        df = scrape_team_stats(code, year)
        if not df.empty:
            all_data.append(df)
        time.sleep(1.0) # サーバー負荷軽減

# 3. 結合と保存
df_player_stats = pd.concat(all_data, ignore_index=True)

# 列の並び替え
cols_order = [
    'year', 'team_name', 'player_name', 'pos', 'games', 'goals', 'assists',
    'cbp_attack', 'cbp_attack_90', 'shoot_point', 'shoot_point_90', 'goal_point', 'goal_point_90'
]
# アシストがない選手のNaNを0で埋める
if 'assists' in df_player_stats.columns:
    df_player_stats['assists'] = df_player_stats['assists'].fillna(0)

df_player_stats = df_player_stats[cols_order]

print("完了しました。")
# CSVとして上書き保存（必要に応じて）
df_player_stats.to_csv('df_player_stats.csv', index=False)
display(df_player_stats.head(10))

Scraping: 鹿島アントラーズ (2021)
Scraping: 浦和レッズ (2021)
Scraping: 柏レイソル (2021)
Scraping: 東京ヴェルディ (2021)
Scraping: 川崎フロンターレ (2021)
Scraping: 清水エスパルス (2021)
Scraping: 京都サンガF.C. (2021)
Scraping: セレッソ大阪 (2021)
Scraping: ファジアーノ岡山 (2021)
Scraping: アビスパ福岡 (2021)
Scraping: 水戸ホーリーホック (2021)
Scraping: ジェフユナイテッド千葉 (2021)
Scraping: ＦＣ東京 (2021)
Scraping: ＦＣ町田ゼルビア (2021)
Scraping: 横浜Ｆ・マリノス (2021)
Scraping: 名古屋グランパス (2021)
Scraping: ガンバ大阪 (2021)
Scraping: ヴィッセル神戸 (2021)
Scraping: サンフレッチェ広島 (2021)
Scraping: Ｖ・ファーレン長崎 (2021)
Scraping: 横浜ＦＣ (2021)
Scraping: 湘南ベルマーレ (2021)
Scraping: アルビレックス新潟 (2021)
Scraping: ジュビロ磐田 (2021)
Scraping: 北海道コンサドーレ札幌 (2021)
Scraping: サガン鳥栖 (2021)
Scraping: 徳島ヴォルティス (2021)
Scraping: 大分トリニータ (2021)
Scraping: ベガルタ仙台 (2021)
Scraping: 鹿島アントラーズ (2022)
Scraping: 浦和レッズ (2022)
Scraping: 柏レイソル (2022)
Scraping: 東京ヴェルディ (2022)
Scraping: 川崎フロンターレ (2022)
Scraping: 清水エスパルス (2022)
Scraping: 京都サンガF.C. (2022)
Scraping: セレッソ大阪 (2022)
Scraping: ファジアーノ岡山 (2022)
Scraping: アビスパ福岡 (2022)
Scraping: 水戸ホーリーホック

,year,team_name,player_name,pos,games,goals,assists,cbp_attack,cbp_attack_90,shoot_point,shoot_point_90,goal_point,goal_point_90
0,2021,鹿島アントラーズ,土居 聖真,MF,36,6,4,40.53,1.59,30.05,1.18,11.21,0.44
1,2021,鹿島アントラーズ,犬飼 智也,DF,29,5,0,26.04,1.10,13.66,0.58,15.11,0.64
2,2021,鹿島アントラーズ,レオ シルバ,MF,29,0,1,31.81,1.62,NaN,NaN,NaN,NaN
3,2021,鹿島アントラーズ,三竿 健斗,MF,34,0,1,50.09,1.74,14.01,0.49,NaN,NaN
4,2021,鹿島アントラーズ,町田 浩樹,DF,34,5,1,33.92,1.00,9.21,0.27,11.15,0.33
5,2021,鹿島アントラーズ,常本 佳吾,DF,26,2,1,35.30,1.49,NaN,NaN,6.88,0.29
6,2021,鹿島アントラーズ,永戸 勝也,DF,29,1,4,35.68,1.57,NaN,NaN,NaN,NaN
7,2021,鹿島アントラーズ,上田 綺世,FW,29,14,0,NaN,NaN,47.66,2.45,35.90,1.84
8,2021,鹿島アントラーズ,荒木 遼太郎,MF,36,10,7,43.71,1.72,34.63,1.36,26.27,1.03
9,2021,鹿島アントラーズ,松村 優太,MF,22,2,0,NaN,NaN,NaN,NaN,8.84,1.05


# 4. チームスタッツ

In [8]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

# 1. 取得する年とスタッツの定義
YEARS = [2020, 2021, 2022, 2023, 2024, 2025]

# 特徴量名とURLパス(value)のマッピング
STATS_MAPPING = {
    'Total Shots': 'shoot',
    'Shots on Target': 'shoot_on_target',
    'Total Shots Against': 'suffer_shoot',
    'Total Goals': 'score',
    'Total Goals Against': 'lost',
    'Clean Sheets': 'clean_sheet',
    'Total Passes': 'pass_count',
    'Total Dribbles': 'dribble_count',
    'Total Through Passes': 'through_pass_count',
    'Total Crosses': 'cross_count',
    'Total Clearances': 'clear_count',
    'Total Tackles': 'tackle_count',
    'Total Blocks': 'block_count',
    'Total Fouls': 'foul_count',
    'Total Interceptions': 'intercept_count',
    'Aerial Duels Won': 'air_battle_win_count',
    'Yellow Cards': 'yellow_count',
    'Red Cards': 'red_count',
    'Average Possession': 'ball_rate',
    'Chances Created': 'chance_create',
    'Duels Won': 'one_on_one',
    'Recoveries': 'recovery_count',
    'Expected Goals': 'expected_goals',
    'Expected Goals Against': 'expected_goals_against',
    'Avg Distance Covered': 'distance_per_game',
    'Avg Sprints': 'sprint_per_game',
    'Distance Covered (In Possession)': 'possession_distance_per_game',
    'Sprints (In Possession)': 'possession_sprint_per_game',
    'Distance Covered (Out of Possession)': 'un_possession_distance_per_game',
    'Sprints (Out of Possession)': 'un_possession_sprint_per_game'
}

def clean_value(val_str):
    """テキストから単位(GOALS, %, kmなど)や改行を完全に除去する"""
    if not val_str:
        return ""
    # 1. 改行や余計な空白を削除
    cleaned = re.sub(r'\s+', '', val_str)
    # 2. 'GOALS' を削除
    cleaned = cleaned.replace('GOALS', '')
    # 3. '%' を削除
    cleaned = cleaned.replace('%', '')
    # 4. 数値のカンマを削除 (例: 25,310 -> 25310)
    cleaned = cleaned.replace(',', '')
    return cleaned

def scrape_jleague_stats():
    all_data = []

    for year in YEARS:
        print(f"=== {year}年のデータを取得中 ===")
        
        # その年の全チームのベースとなる辞書（後で結合するため）
        yearly_team_stats = {}
        
        for stat_name, stat_path in STATS_MAPPING.items():
            url = f"https://www.jleague.jp/stats/j1/club/{year}/{stat_path}/"
            
            try:
                r = requests.get(url, timeout=10)
                r.raise_for_status()
                soup = BeautifulSoup(r.content, 'html.parser')
                
                # ランキングリストの抽出
                ranking_list = soup.find('ul', class_='ranking_list')
                if not ranking_list:
                    print(f"Warning: {year}年の {stat_name} データが見つかりません。")
                    time.sleep(1)
                    continue
                
                # 各チームのデータを抽出
                for li in ranking_list.find_all('li', recursive=False):
                    team_tag = li.find('p', class_='team')
                    stats_tag = li.find('div', class_=re.compile(r'ranking_stats'))
                    
                    if team_tag and stats_tag:
                        team_name = team_tag.text.strip()
                        # 値の部分（spanタグ等の除去）
                        val_str = stats_tag.find('p').text
                        val_clean = clean_value(val_str)
                        
                        # 辞書に格納
                        if team_name not in yearly_team_stats:
                            yearly_team_stats[team_name] = {'year': year, 'team_name': team_name}
                            
                        yearly_team_stats[team_name][stat_name] = val_clean
                        
            except Exception as e:
                print(f"Error skipping {stat_name} ({year}): {e}")
            
            # サーバーへの負荷軽減（0.5秒待機）
            time.sleep(0.5)
            
        # その年の辞書をリストに追加
        all_data.extend(list(yearly_team_stats.values()))
        
    # DataFrameに変換
    df = pd.DataFrame(all_data)
    
    # 列の順序を整える
    cols = ['year', 'team_name'] + list(STATS_MAPPING.keys())
    # 取得できなかった列が存在する可能性を考慮
    existing_cols = [c for c in cols if c in df.columns]
    df = df[existing_cols]
    
    return df

# 実行
df_team_stats = scrape_jleague_stats()
# 結果の確認
display(df_team_stats.head())

=== 2020年のデータを取得中 ===
=== 2021年のデータを取得中 ===
=== 2022年のデータを取得中 ===
=== 2023年のデータを取得中 ===
=== 2024年のデータを取得中 ===
=== 2025年のデータを取得中 ===


,year,team_name,Total Shots,Shots on Target,Total Shots Against,Total Goals,Total Goals Against,Clean Sheets,Total Passes,Total Dribbles,...,Duels Won,Recoveries,Expected Goals,Expected Goals Against,Avg Distance Covered,Avg Sprints,Distance Covered (In Possession),Sprints (In Possession),Distance Covered (Out of Possession),Sprints (Out of Possession)
0,2020,川崎フロンターレ,643,240,348,88,31,11,21047,411,...,681,1153,82.2,35.0,111,157,44,74,46,82
1,2020,鹿島アントラーズ,597,183,374,55,44,6,17063,389,...,677,1287,61.0,42.6,112,167,42,80,45,86
2,2020,横浜Ｆ・マリノス,534,176,431,69,59,6,22769,435,...,661,1161,61.5,46.1,121,184,51,89,47,93
3,2020,北海道コンサドーレ札幌,494,143,434,47,58,8,18201,556,...,736,1116,53.1,51.8,114,165,43,75,47,88
4,2020,ヴィッセル神戸,482,162,438,50,59,4,20609,364,...,609,1069,51.4,48.1,108,163,44,73,44,89


# 5. コンディション・外部要因（Conditioning & Context）

In [9]:
import pandas as pd
from geopy.distance import geodesic
import numpy as np

# Jリーグの略称からWikipediaのクラブ名へのマッピング（前回のものを継続）
TEAM_NAME_MAP = {
    '札幌': '北海道コンサドーレ札幌', '鹿島': '鹿島アントラーズ', '浦和': '浦和レッズ',
    '柏': '柏レイソル', 'FC東京': 'FC東京', '東京Ｖ': '東京ヴェルディ', '町田': 'FC町田ゼルビア',
    '川崎Ｆ': '川崎フロンターレ', '横浜FM': '横浜F・マリノス', '横浜FC': '横浜FC',
    '湘南': '湘南ベルマーレ', '新潟': 'アルビレックス新潟', '清水': '清水エスパルス',
    '磐田': 'ジュビロ磐田', '名古屋': '名古屋グランパス', '京都': '京都サンガF.C.',
    'Ｇ大阪': 'ガンバ大阪', 'Ｃ大阪': 'セレッソ大阪', '神戸': 'ヴィッセル神戸',
    '広島': 'サンフレッチェ広島', '徳島': '徳島ヴォルティス', '福岡': 'アビスパ福岡',
    '鳥栖': 'サガン鳥栖', '大分': '大分トリニータ', '仙台': 'ベガルタ仙台'
}

import numpy as np

def calculate_rest_days(df):
    df_copy = df.copy()
    team_matches = {}
    
    # チームごとの試合日程をリストアップ
    for _, row in df_copy.iterrows():
        date = row['Date_Parsed']
        home_team, away_team = row['Home'], row['Away']
        
        if home_team not in team_matches: team_matches[home_team] = []
        if away_team not in team_matches: team_matches[away_team] = []
        
        team_matches[home_team].append({'date': date, 'is_home': True})
        team_matches[away_team].append({'date': date, 'is_home': False})
    
    # 日付順にソート
    for team in team_matches:
        team_matches[team].sort(key=lambda x: x['date'])
    
    rest_days_home, rest_days_away = [], []
    
    # 休養日数の計算
    for _, row in df_copy.iterrows():
        date, home_team, away_team = row['Date_Parsed'], row['Home'], row['Away']
        
        # ホームチームの休養日数
        prev_home_date = None
        for match in team_matches[home_team]:
            if match['date'] == date and match['is_home']:
                rest_days_home.append((date - prev_home_date).days if prev_home_date else np.nan)
                break
            prev_home_date = match['date']
            
        # アウェイチームの休養日数
        prev_away_date = None
        for match in team_matches[away_team]:
            if match['date'] == date and not match['is_home']:
                rest_days_away.append((date - prev_away_date).days if prev_away_date else np.nan)
                break
            prev_away_date = match['date']
            
    df_copy['Home_Rest_Days'] = rest_days_home
    df_copy['Away_Rest_Days'] = rest_days_away
    
    # ==========================================
    # 修正部分：第1節と初期のNullを処理する
    # ==========================================
    # 正規表現を用いて「第1節」または「第１節」（全角半角どちらでも）を含む行を特定
    is_first_section = df_copy['Section'].astype(str).str.contains(r'第[1１]節')
    
    # 第1節の休養日数を強制的に0.0にリセット（シーズン跨ぎの計算を上書き）
    df_copy.loc[is_first_section, 'Home_Rest_Days'] = 0.0
    df_copy.loc[is_first_section, 'Away_Rest_Days'] = 0.0
    
    # 2021年の第1節など、前節がないことによる NaN も 0.0 で穴埋め
    df_copy['Home_Rest_Days'] = df_copy['Home_Rest_Days'].fillna(0.0)
    df_copy['Away_Rest_Days'] = df_copy['Away_Rest_Days'].fillna(0.0)
    
    return df_copy

def calculate_travel_distance_fixed(df_fixtures, stadium_df):
    """
    すべての節において、アウェイチームの本拠地からホームチームの本拠地までの距離を計算する
    """
    df_copy = df_fixtures.copy()
    
    # Wikipediaデータから「クラブ名 -> (緯度, 経度)」の辞書を作成
    club_coords = {}
    for _, row in stadium_df.iterrows():
        if pd.notna(row['Latitude']) and pd.notna(row['Longitude']):
            club_coords[row['Club']] = (row['Latitude'], row['Longitude'])
    
    # 座標取得のヘルパー関数
    def get_coords(team_short_name):
        wiki_name = TEAM_NAME_MAP.get(team_short_name)
        return club_coords.get(wiki_name)

    travel_distances = []

    for _, row in df_copy.iterrows():
        home_team = row['Home']
        away_team = row['Away']
        
        # アウェイチームの本拠地座標
        coords_away_base = get_coords(away_team)
        # ホームチームの本拠地座標（＝今回の試合会場）
        coords_home_base = get_coords(home_team)
        
        if coords_away_base and coords_home_base:
            # 本拠地間の直線距離を計算
            dist = geodesic(coords_away_base, coords_home_base).km
            travel_distances.append(round(dist, 1))
        else:
            # 座標が見つからない場合は暫定的に0（または平均的な遠征距離）を入れる
            travel_distances.append(0.0)
            
    df_copy['Away_Travel_Distance_km'] = travel_distances
    
    return df_copy

# === 実行 ===
df_fixtures = calculate_rest_days(df_fixtures)
df_fixtures = calculate_travel_distance_fixed(df_fixtures, stadium_df)

# 結果の確認（第1節から数値が入るようになります）
print("中日、移動距離データ:")
display(df_fixtures[['Date_Parsed', 'Home', 'Away', 'Home_Rest_Days', 'Away_Rest_Days', 'Away_Travel_Distance_km']].head(15))

中日、移動距離データ:


,Date_Parsed,Home,Away,Home_Rest_Days,Away_Rest_Days,Away_Travel_Distance_km
0,2021-02-26,川崎Ｆ,横浜FM,0.0,0.0,6.7
1,2021-02-27,浦和,FC東京,0.0,0.0,27.7
2,2021-02-27,札幌,横浜FC,0.0,0.0,852.1
3,2021-02-27,大分,徳島,0.0,0.0,296.6
4,2021-02-27,広島,仙台,0.0,0.0,873.1
5,2021-02-27,鹿島,清水,0.0,0.0,222.2
6,2021-02-27,湘南,鳥栖,0.0,0.0,841.7
7,2021-02-27,Ｃ大阪,柏,0.0,0.0,427.7
8,2021-02-27,神戸,Ｇ大阪,0.0,0.0,33.3
9,2021-02-28,福岡,名古屋,0.0,0.0,642.0


In [10]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import datetime

def scrape_weather_data():
    """
    Football Labから各チームのホームゲームの天候を取得し、
    (日付, ホームチーム名) をキーとした辞書を返す
    """
    weather_dict = {}
    
    # ユーザーエージェントを設定
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    
    print("=== 天候データのスクレイピングを開始します ===")
    for year in years:
        for code, short_name in TEAMS.items():
            url = f"https://www.football-lab.jp/{code}/match?year={year}"
            
            try:
                r = requests.get(url, headers=headers, timeout=10)
                if r.status_code != 200:
                    continue # ページがない場合（J2降格年など）はスキップ
                    
                soup = BeautifulSoup(r.content, 'html.parser')
                
                # statsTbl10 クラスのテーブルを取得
                table = soup.find('table', class_='statsTbl10')
                if not table:
                    continue
                    
                rows = table.find('tbody').find_all('tr')
                for row in rows:
                    cols = row.find_all('td')
                    
                    # 列数が足りない行はスキップ
                    if len(cols) < 9:
                        continue
                        
                    # Home/Awayの判定 (Hの試合のみ取得して重複を防ぐ)
                    h_a = cols[5].get_text(strip=True)
                    if h_a != 'H':
                        continue
                        
                    # 日付の取得 (data-order="20250215" -> "2025-02-15")
                    date_td = cols[1]
                    if 'data-order' in date_td.attrs:
                        date_raw = date_td['data-order']
                        # "YYYYMMDD" を "YYYY-MM-DD" に変換
                        if len(date_raw) == 8:
                            date_str = f"{date_raw[:4]}-{date_raw[4:6]}-{date_raw[6:]}"
                            
                            # 天候の取得
                            weather = cols[8].get_text(strip=True)
                            
                            # 辞書に格納: key = (日付文字列, ホームチーム名)
                            weather_dict[(date_str, short_name)] = weather
                            
            except Exception as e:
                print(f"Error scraping {short_name} ({year}): {e}")
                
            time.sleep(0.5) # サーバー負荷軽減
            
    return weather_dict

# 1. スレイピングの実行
weather_dict = scrape_weather_data()
print(f"取得した天候データ: {len(weather_dict)}件")

# 2. df_fixturesへの追加処理
def add_weather_to_fixtures(df, weather_dict):
    df_copy = df.copy()

    # フルネームから略称へのマッピング（df_fixturesの'Home'列と一致させるため）
    FULL_TO_SHORT = {
        '鹿島アントラーズ': '鹿島', '浦和レッズ': '浦和', '柏レイソル': '柏', 
        '東京ヴェルディ': '東京Ｖ', '川崎フロンターレ': '川崎Ｆ', '清水エスパルス': '清水', 
        '京都サンガF.C.': '京都', 'セレッソ大阪': 'Ｃ大阪', 'アビスパ福岡': '福岡', 
        'ＦＣ東京': 'FC東京', 'ＦＣ町田ゼルビア': '町田', '横浜Ｆ・マリノス': '横浜FM', 
        '名古屋グランパス': '名古屋', 'ガンバ大阪': 'Ｇ大阪', 'ヴィッセル神戸': '神戸', 
        'サンフレッチェ広島': '広島', '横浜ＦＣ': '横浜FC', '湘南ベルマーレ': '湘南', 
        'アルビレックス新潟': '新潟', 'ジュビロ磐田': '磐田', '北海道コンサドーレ札幌': '札幌', 
        'サガン鳥栖': '鳥栖', '徳島ヴォルティス': '徳島', '大分トリニータ': '大分', 
        'ベガルタ仙台': '仙台'
    }
    
    # 1. ここで「略称」をキーにした新しい辞書を作っています
    new_weather_dict = {}
    for (d_str, full_name), w in weather_dict.items():
        short_name = FULL_TO_SHORT.get(full_name, full_name) 
        new_weather_dict[(d_str, short_name)] = w
    
    # 日付形式の統一
    if pd.api.types.is_datetime64_any_dtype(df_copy['Date_Parsed']):
        date_strs = df_copy['Date_Parsed'].dt.strftime('%Y-%m-%d')
    else:
        date_strs = df_copy['Date_Parsed'].astype(str)
        
    weathers = []
    for d_str, h_team in zip(date_strs, df_copy['Home']):
        # 【重要】ここを new_weather_dict に修正しました！
        w = new_weather_dict.get((d_str, h_team), np.nan)
        weathers.append(w)
        
    df_copy['Weather'] = weathers
    return df_copy

# === 実行 ===
df_fixtures = add_weather_to_fixtures(df_fixtures, weather_dict)

# 確認
print("天候を追加したdf_fixtures:")
display(df_fixtures[['Date_Parsed', 'Home', 'Away', 'Weather']].head(10))

=== 天候データのスクレイピングを開始します ===
取得した天候データ: 2745件
天候を追加したdf_fixtures:


,Date_Parsed,Home,Away,Weather
0,2021-02-26,川崎Ｆ,横浜FM,曇
1,2021-02-27,浦和,FC東京,晴
2,2021-02-27,札幌,横浜FC,屋内
3,2021-02-27,大分,徳島,曇
4,2021-02-27,広島,仙台,晴
5,2021-02-27,鹿島,清水,晴
6,2021-02-27,湘南,鳥栖,晴
7,2021-02-27,Ｃ大阪,柏,晴
8,2021-02-27,神戸,Ｇ大阪,晴
9,2021-02-28,福岡,名古屋,晴


# 6. 戦術・相性（Tactical Interaction）

In [11]:
# シーズンごとの各フォーメーション
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time


def scrape_main_formations():
    results = []
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    
    print("=== フォーメーションデータのスクレイピングを開始します ===")
    
    for year in YEARS:
        for code, full_name in TEAMS.items():
            url = f"https://www.football-lab.jp/{code}/formation?year={year}"
            
            try:
                r = requests.get(url, headers=headers, timeout=10)
                if r.status_code != 200:
                    continue
                
                soup = BeautifulSoup(r.content, 'html.parser')
                # デスクトップ版の統計テーブルを特定
                table = soup.find('table', class_='statsTbl dsktp')
                if not table:
                    continue
                
                rows = table.find('tbody').find_all('tr')
                
                formation_data = []
                for row in rows:
                    cols = row.find_all('td')
                    if len(cols) < 2:
                        continue
                    
                    system_name = cols[0].get_text(strip=True)
                    # 「合計」行は除外
                    if system_name == "合計":
                        continue
                        
                    match_count_str = cols[1].get_text(strip=True)
                    match_count = int(match_count_str) if match_count_str.isdigit() else 0
                    
                    formation_data.append({
                        'Formation': system_name,
                        'MatchCount': match_count
                    })
                
                if formation_data:
                    # 試合数が最も多いフォーメーションを選択
                    main_formation = max(formation_data, key=lambda x: x['MatchCount'])['Formation']
                    
                    results.append({
                        'Year': year,
                        'Team': full_name,
                        'TeamCode': code,
                        'Main_Formation': main_formation
                    })
                    print(f"  [Success] {year} {full_name}: {main_formation}")
                
            except Exception as e:
                print(f"  [Error] {year} {code}: {e}")
                
            time.sleep(0.5) # サーバー負荷軽減
            
    return pd.DataFrame(results)

# 1. スクレイピング実行
df_formations = scrape_main_formations()

# 2. 結果の確認
print("\n取得結果（先頭10件）:")
print(df_formations.head(10))

# 既存の df_fixtures への結合例（必要に応じて）
# df_fixtures = df_fixtures.merge(df_formations, left_on=['Year', 'Home'], right_on=['Year', 'Team'], how='left')

=== フォーメーションデータのスクレイピングを開始します ===
  [Success] 2020 鹿島アントラーズ: 4-4-2
  [Success] 2020 浦和レッズ: 4-4-2
  [Success] 2020 柏レイソル: 4-2-3-1
  [Success] 2020 東京ヴェルディ: 4-1-2-3
  [Success] 2020 川崎フロンターレ: 4-1-2-3
  [Success] 2020 清水エスパルス: 4-2-3-1
  [Success] 2020 京都サンガF.C.: 3-3-2-2
  [Success] 2020 セレッソ大阪: 4-4-2
  [Success] 2020 ファジアーノ岡山: 4-4-2
  [Success] 2020 アビスパ福岡: 4-4-2
  [Success] 2020 水戸ホーリーホック: 4-4-2
  [Success] 2020 ジェフユナイテッド千葉: 4-4-2
  [Success] 2020 ＦＣ東京: 4-1-2-3
  [Success] 2020 ＦＣ町田ゼルビア: 4-4-2
  [Success] 2020 横浜Ｆ・マリノス: 4-2-3-1
  [Success] 2020 名古屋グランパス: 4-2-3-1
  [Success] 2020 ガンバ大阪: 4-4-2
  [Success] 2020 ヴィッセル神戸: 4-1-2-3
  [Success] 2020 サンフレッチェ広島: 3-4-2-1
  [Success] 2020 Ｖ・ファーレン長崎: 4-4-2
  [Success] 2020 横浜ＦＣ: 4-4-2
  [Success] 2020 湘南ベルマーレ: 3-3-2-2
  [Success] 2020 アルビレックス新潟: 4-2-3-1
  [Success] 2020 ジュビロ磐田: 3-4-1-2
  [Success] 2020 北海道コンサドーレ札幌: 3-4-2-1
  [Success] 2020 サガン鳥栖: 4-4-2
  [Success] 2020 徳島ヴォルティス: 3-4-2-1
  [Success] 2020 大分トリニータ: 3-4-2-1
  [Success] 2020 ベガルタ仙台: 4-1-2

In [12]:
import pandas as pd
import numpy as np

# --- 1. 型の統一 ---
# Seasonを数値型に変換（"2021" -> 2021）
df_fixtures['Season'] = pd.to_numeric(df_fixtures['Season'], errors='coerce')
# Yearも念のため数値型に
df_formations['Year'] = pd.to_numeric(df_formations['Year'], errors='coerce')

# --- 2. チーム名の名寄せ用辞書 ---
FULL_TO_SHORT = {
    '鹿島アントラーズ': '鹿島', '浦和レッズ': '浦和', '柏レイソル': '柏', 
    '東京ヴェルディ': '東京Ｖ', '川崎フロンターレ': '川崎Ｆ', '清水エスパルス': '清水', 
    '京都サンガF.C.': '京都', 'セレッソ大阪': 'Ｃ大阪', 'ファジアーノ岡山': '岡山',
    'アビスパ福岡': '福岡', '水戸ホーリーホック': '水戸', 'ジェフユナイテッド千葉': '千葉',
    'ＦＣ東京': 'FC東京', 'ＦＣ町田ゼルビア': '町田', '横浜Ｆ・マリノス': '横浜FM', 
    '名古屋グランパス': '名古屋', 'ガンバ大阪': 'Ｇ大阪', 'ヴィッセル神戸': '神戸', 
    'サンフレッチェ広島': '広島', 'Ｖ・ファーレン長崎': '長崎', '横浜ＦＣ': '横浜FC', 
    '湘南ベルマーレ': '湘南', 'アルビレックス新潟': '新潟', 'ジュビロ磐田': '磐田', 
    '北海道コンサドーレ札幌': '札幌', 'サガン鳥栖': '鳥栖', '徳島ヴォルティス': '徳島', 
    '大分トリニータ': '大分', 'ベガルタ仙台': '仙台'
}

# 3. 略称カラムの作成
df_formations['Team_Short'] = df_formations['Team'].map(FULL_TO_SHORT)

# --- 4. 結合処理 ---

# ホームチームのフォーメーションを結合
df_fixtures = pd.merge(
    df_fixtures, 
    df_formations[['Year', 'Team_Short', 'Main_Formation']], 
    left_on=['Season', 'Home'], 
    right_on=['Year', 'Team_Short'], 
    how='left'
).rename(columns={'Main_Formation': 'Home_Formation'}).drop(['Year', 'Team_Short'], axis=1)

# アウェイチームのフォーメーションを結合
df_fixtures = pd.merge(
    df_fixtures, 
    df_formations[['Year', 'Team_Short', 'Main_Formation']], 
    left_on=['Season', 'Away'], 
    right_on=['Year', 'Team_Short'], 
    how='left'
).rename(columns={'Main_Formation': 'Away_Formation'}).drop(['Year', 'Team_Short'], axis=1)

# 結果の確認
display(df_fixtures[['Season', 'Home', 'Away', 'Home_Formation', 'Away_Formation']].head())

,Season,Home,Away,Home_Formation,Away_Formation
0,2021,川崎Ｆ,横浜FM,4-1-2-3,4-2-3-1
1,2021,浦和,FC東京,4-2-3-1,4-2-3-1
2,2021,札幌,横浜FC,3-4-2-1,3-4-2-1
3,2021,大分,徳島,3-4-2-1,4-2-3-1
4,2021,広島,仙台,3-4-2-1,4-4-2


In [13]:
# シーズン別の得点パターンをスクレイピングする関数
def scrape_seasonal_goal_patterns(years):
    all_data = []
    headers = {"User-Agent": "Mozilla/5.0"}
    
    # ユーザー指定の特徴量名
    feature_names = [
        "Goal PK", "Goal Direct Set Piece", "Goal Indirect Set Piece", 
        "Goal Cross", "Goal Through Pass", "Goal from pass within 30m", 
        "Goal from pass farther than 30m", "Goal from Dribble", 
        "Goal from Ball Lost", "Goal Other"
    ]

    print("=== シーズン別得点パターンの取得を開始します ===")
    
    for year in years:
        # J1とJ2の両カテゴリーをチェック（昇降格に対応するため）
        for category in ['j1', 'j2']:
            url = f"https://www.football-lab.jp/summary/team_ranking/{category}?year={year}&data=goal"
            try:
                r = requests.get(url, headers=headers, timeout=10)
                if r.status_code != 200:
                    continue
                
                soup = BeautifulSoup(r.content, 'html.parser')
                scripts = soup.find_all('script')
                
                target_script = ""
                for s in scripts:
                    if s.string and "google.visualization.arrayToDataTable" in s.string:
                        target_script = s.string
                        break
                
                if not target_script:
                    continue

                # JS内の配列データを抽出
                pattern = re.compile(r"\[['\"](.+?)['\"],\s*([\d,\s]+)\]")
                matches = pattern.findall(target_script)
                
                for team_name, values_str in matches:
                    if team_name == 'チーム':
                        continue
                    
                    values = [int(v.strip()) for v in values_str.split(',')]
                    
                    row = {
                        'Season': int(year),
                        'Category': category.upper(),
                        'Team_Short': team_name
                    }
                    # 特徴量を追加
                    for i, feat in enumerate(feature_names):
                        row[feat] = values[i]
                        
                    all_data.append(row)
                
                print(f"  [Done] {year} {category.upper()}")
                
            except Exception as e:
                print(f"  [Error] {year} {category}: {e}")
            
            time.sleep(0.5)
            
    # データフレーム化
    df = pd.DataFrame(all_data)
    
    # 重複削除（J1/J2両方にデータがある場合の予備策）
    df = df.drop_duplicates(subset=['Season', 'Team_Short'], keep='first')
    
    return df

# === 実行 ===
YEARS = [2020, 2021, 2022, 2023, 2024, 2025]
df_goal_patterns = scrape_seasonal_goal_patterns(YEARS)

# === データの確認 ===
print("\n作成された得点パターンデータフレーム:")
display(df_goal_patterns.head(10))

=== シーズン別得点パターンの取得を開始します ===
  [Done] 2020 J1
  [Done] 2020 J2
  [Done] 2021 J1
  [Done] 2021 J2
  [Done] 2022 J1
  [Done] 2022 J2
  [Done] 2023 J1
  [Done] 2023 J2
  [Done] 2024 J1
  [Done] 2024 J2
  [Done] 2025 J1
  [Done] 2025 J2

作成された得点パターンデータフレーム:


,Season,Category,Team_Short,Goal PK,Goal Direct Set Piece,Goal Indirect Set Piece,Goal Cross,Goal Through Pass,Goal from pass within 30m,Goal from pass farther than 30m,Goal from Dribble,Goal from Ball Lost,Goal Other
0,2020,J1,川崎Ｆ,7,1,16,15,8,23,2,4,8,4
1,2020,J1,横浜FM,2,0,10,16,3,19,2,8,7,2
2,2020,J1,柏,1,0,10,10,7,12,5,7,6,2
3,2020,J1,鹿島,1,0,9,14,2,11,3,2,9,4
4,2020,J1,神戸,2,0,10,13,6,7,3,6,2,1
5,2020,J1,清水,1,0,15,10,4,10,0,2,3,3
6,2020,J1,札幌,1,3,4,10,7,12,2,2,3,3
7,2020,J1,FC東京,4,3,9,8,8,7,0,2,4,2
8,2020,J1,広島,2,2,7,5,7,11,5,2,4,1
9,2020,J1,Ｇ大阪,5,1,9,6,1,12,1,3,5,3


In [14]:
# チームスタイル指標をスクレイピングする関数
def scrape_team_styles(years):
    all_data = []
    headers = {"User-Agent": "Mozilla/5.0"}
    
    # 抽出する指標名とカラム名の対応
    column_mapping = {
        0: "Set-piece Index",
        1: "Left Attack Index",
        2: "Center Attack Index",
        3: "Right Attack Index",
        4: "Short Counter",
        5: "Long Counter",
        6: "Opponent Area Possession",
        7: "My Area Possession"
    }

    print("=== チームスタイル指標の取得を開始します ===")
    
    for year in years:
        for category in ['j1', 'j2']:
            # data=21 はチームスタイル指数のページ
            url = f"https://www.football-lab.jp/summary/team_style/{category}?year={year}&data=21"
            
            try:
                r = requests.get(url, headers=headers, timeout=10)
                if r.status_code != 200:
                    continue
                
                soup = BeautifulSoup(r.content, 'html.parser')
                table = soup.find('table', id='tstyle2')
                
                if not table:
                    continue
                
                rows = table.find('tbody').find_all('tr')
                
                for row in rows:
                    cols = row.find_all('td')
                    if len(cols) < 10:
                        continue
                    
                    # チーム名は2番目のtd (index 1)
                    team_name = cols[1].get_text(strip=True)
                    
                    # 各指数を取得 (index 2以降)
                    style_values = []
                    for i in range(2, 10):
                        val_str = cols[i].get_text(strip=True)
                        val = int(val_str) if val_str.isdigit() else 0
                        style_values.append(val)
                    
                    # データの格納
                    res_row = {
                        'Season': int(year),
                        'Category': category.upper(),
                        'Team_Short': team_name
                    }
                    
                    for idx, name in column_mapping.items():
                        res_row[name] = style_values[idx]
                        
                    all_data.append(res_row)
                
                print(f"  [Done] {year} {category.upper()}")
                
            except Exception as e:
                print(f"  [Error] {year} {category}: {e}")
            
            time.sleep(0.5)
            
    df = pd.DataFrame(all_data)
    # 重複の排除
    df = df.drop_duplicates(subset=['Season', 'Team_Short'], keep='first')
    return df

# === 実行 ===
YEARS = [2020, 2021, 2022, 2023, 2024, 2025]
df_team_styles = scrape_team_styles(YEARS)

# === 確認 ===
print("\n作成されたチームスタイル指標データフレーム:")
display(df_team_styles.head())

=== チームスタイル指標の取得を開始します ===
  [Done] 2020 J1
  [Done] 2020 J2
  [Done] 2021 J1
  [Done] 2021 J2
  [Done] 2022 J1
  [Done] 2022 J2
  [Done] 2023 J1
  [Done] 2023 J2
  [Done] 2024 J1
  [Done] 2024 J2
  [Done] 2025 J1
  [Done] 2025 J2

作成されたチームスタイル指標データフレーム:


,Season,Category,Team_Short,Set-piece Index,Left Attack Index,Center Attack Index,Right Attack Index,Short Counter,Long Counter,Opponent Area Possession,My Area Possession
0,2020,J1,川崎Ｆ,61,63,59,57,61,41,73,48
1,2020,J1,Ｇ大阪,49,44,52,65,49,44,55,45
2,2020,J1,名古屋,52,59,37,45,47,42,46,48
3,2020,J1,Ｃ大阪,46,40,45,39,34,40,45,58
4,2020,J1,鹿島,72,48,62,69,65,62,55,40


# 7. 戦術・相性（Tactical Interaction）

In [15]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def scrape_standings_and_create_features():
    data = []
    headers = {"User-Agent": "Mozilla/5.0"}
    
    print("=== 順位・勝ち点データの取得を開始します ===")
    
    # 1. スクレイピング処理
    for year, comp_id in year_id_map.items():
        for section in range(1, 39):  # 第1節〜第38節
            url = f"https://data.j-league.or.jp/SFRT01/?yearId={year}&competitionId={comp_id}&competitionSectionId={section}&search=search"
            
            try:
                r = requests.get(url, headers=headers, timeout=10)
                if r.status_code != 200:
                    continue
                
                soup = BeautifulSoup(r.content, 'html.parser')
                table = soup.find('table', id='search_result')
                
                if not table:
                    continue
                
                rows = table.find('tbody').find_all('tr')
                for row in rows:
                    cols = row.find_all('td')
                    if len(cols) < 5:
                        continue
                    
                    # htmlのクラス指定からデータを抽出
                    rank_str = cols[1].get_text(strip=True)
                    team_name = cols[2].get_text(strip=True)
                    points_str = cols[3].get_text(strip=True)
                    
                    if not rank_str.isdigit() or not points_str.isdigit():
                        continue
                        
                    data.append({
                        'Year': year,
                        'Section': section,
                        'Team': team_name,
                        'Rank': int(rank_str),
                        'Points': int(points_str)
                    })
                    
            except Exception as e:
                print(f"  [Error] {year} 第{section}節: {e}")
                
            time.sleep(0.5) # サーバー負荷軽減
            
        print(f"  [Done] {year}年のデータを取得しました")
    
    print("\n=== 特徴量（Season Progress / Urgency Score）の作成を開始します ===")
    df = pd.DataFrame(data)

    
    # 2. Season Progress の作成 (0.026 ~ 1.0)
    df['Season_Progress'] = df['Section'] / 38.0
    
    # 3. Urgency Score の作成
    # 年・節ごとの「1位の勝ち点」を取得
    top_points = df.groupby(['Year', 'Section'])['Points'].max().reset_index(name='Points_1st')
    
    # 年・節ごとの「18位(降格ライン)の勝ち点」を取得
    # ※18位が同率で複数いる場合や、データ欠損を考慮して「18位以上の最低勝ち点」ではなく「18位以下の最大勝ち点」を取得
    rank18_points = df[df['Rank'] >= 18].groupby(['Year', 'Section'])['Points'].max().reset_index(name='Points_18th')
    
    # 結合
    df = pd.merge(df, top_points, on=['Year', 'Section'], how='left')
    df = pd.merge(df, rank18_points, on=['Year', 'Section'], how='left')
    
    # 勝ち点差の計算（自分が1位/18位以下の場合は0にするため clip(lower=0) を使用）
    df['Diff_1st'] = (df['Points_1st'] - df['Points']).clip(lower=0)
    df['Diff_18th'] = (df['Points'] - df['Points_18th']).clip(lower=0)
    
    # 1位と18位、近い方の勝ち点差を採用（序盤で18位がいない場合は1位との差のみを使用）
    df['Dist_to_Edge'] = df[['Diff_1st', 'Diff_18th']].min(axis=1, skipna=True)
    
    # 計算ロジック: (1 / (勝ち点差 + 1)) * (Season_Progressの2乗)
    df['Urgency_Score'] = (1 / (df['Dist_to_Edge'] + 1)) * (df['Season_Progress'] ** 2)
    
    # 中間生成した不要なカラムを削除
    df = df.drop(columns=['Points_1st', 'Points_18th', 'Diff_1st', 'Diff_18th', 'Dist_to_Edge'])
    
    # ソートして整理
    df = df.sort_values(['Year', 'Section', 'Rank']).reset_index(drop=True)
    
    return df

# 実行
df_season_stats = scrape_standings_and_create_features()

# 結果の確認
print("\n完成したデータフレーム (第30節付近のサンプル):")
display(df_season_stats[df_season_stats['Section'] == 30].head(10))

=== 順位・勝ち点データの取得を開始します ===
  [Done] 2021年のデータを取得しました
  [Done] 2022年のデータを取得しました
  [Done] 2023年のデータを取得しました
  [Done] 2024年のデータを取得しました
  [Done] 2025年のデータを取得しました

=== 特徴量（Season Progress / Urgency Score）の作成を開始します ===

完成したデータフレーム (第30節付近のサンプル):


,Year,Section,Team,Rank,Points,Season_Progress,Urgency_Score
580,2021,30,川崎フロンターレ,1,75,0.789474,0.623269
581,2021,30,横浜Ｆ・マリノス,2,66,0.789474,0.062327
582,2021,30,名古屋グランパス,3,57,0.789474,0.032804
583,2021,30,ヴィッセル神戸,4,54,0.789474,0.028330
584,2021,30,浦和レッズ,5,54,0.789474,0.028330
585,2021,30,鹿島アントラーズ,6,53,0.789474,0.027099
586,2021,30,サガン鳥栖,7,51,0.789474,0.024931
587,2021,30,アビスパ福岡,8,46,0.789474,0.024931
588,2021,30,ＦＣ東京,9,46,0.789474,0.024931
589,2021,30,サンフレッチェ広島,10,42,0.789474,0.029679


In [16]:
# 結果の確認
print("\n完成したデータフレーム (第30節付近のサンプル):")
display(df_season_stats[df_season_stats['Section'] == 30].head(20))


完成したデータフレーム (第30節付近のサンプル):


,Year,Section,Team,Rank,Points,Season_Progress,Urgency_Score
580,2021,30,川崎フロンターレ,1,75,0.789474,0.623269
581,2021,30,横浜Ｆ・マリノス,2,66,0.789474,0.062327
582,2021,30,名古屋グランパス,3,57,0.789474,0.032804
583,2021,30,ヴィッセル神戸,4,54,0.789474,0.028330
584,2021,30,浦和レッズ,5,54,0.789474,0.028330
585,2021,30,鹿島アントラーズ,6,53,0.789474,0.027099
586,2021,30,サガン鳥栖,7,51,0.789474,0.024931
587,2021,30,アビスパ福岡,8,46,0.789474,0.024931
588,2021,30,ＦＣ東京,9,46,0.789474,0.024931
589,2021,30,サンフレッチェ広島,10,42,0.789474,0.029679


# 8 CSVにてRawDataを保存

In [17]:
import os

# Define the directory path
directory = '/Users/akihirookuyama/Soccer_Score_App/Data/RawData'

# Ensure the directory exists
os.makedirs(directory, exist_ok=True)

# List of dataframes and their names
dataframes = {
    'df_fixtures': df_fixtures,
    'df_managers': df_managers,
    'df_manager_changes': df_manager_changes,
    'df_ranks': df_ranks,
    'df_market_values': df_market_values,
    'df_xg': df_xg,
    'df_agi_kagi': df_agi_kagi,
    'stadium_df': stadium_df,
    'df_player_stats': df_player_stats,
    'df_team_stats': df_team_stats,
    'df_goal_patterns': df_goal_patterns,
    'df_team_styles': df_team_styles,
    'df_season_stats': df_season_stats
}

# Save each dataframe as CSV
for name, df in dataframes.items():
    file_path = os.path.join(directory, f'{name}.csv')
    df.to_csv(file_path, index=False, encoding='utf-8-sig')
    print(f'Saved {name} to {file_path}')

Saved df_fixtures to /Users/akihirookuyama/Soccer_Score_App/Data/RawData/df_fixtures.csv
Saved df_managers to /Users/akihirookuyama/Soccer_Score_App/Data/RawData/df_managers.csv
Saved df_manager_changes to /Users/akihirookuyama/Soccer_Score_App/Data/RawData/df_manager_changes.csv
Saved df_ranks to /Users/akihirookuyama/Soccer_Score_App/Data/RawData/df_ranks.csv
Saved df_market_values to /Users/akihirookuyama/Soccer_Score_App/Data/RawData/df_market_values.csv
Saved df_xg to /Users/akihirookuyama/Soccer_Score_App/Data/RawData/df_xg.csv
Saved df_agi_kagi to /Users/akihirookuyama/Soccer_Score_App/Data/RawData/df_agi_kagi.csv
Saved stadium_df to /Users/akihirookuyama/Soccer_Score_App/Data/RawData/stadium_df.csv
Saved df_player_stats to /Users/akihirookuyama/Soccer_Score_App/Data/RawData/df_player_stats.csv
Saved df_team_stats to /Users/akihirookuyama/Soccer_Score_App/Data/RawData/df_team_stats.csv
Saved df_goal_patterns to /Users/akihirookuyama/Soccer_Score_App/Data/RawData/df_goal_patterns